In [0]:
from pyspark.sql.functions import col, upper, trim, to_date, hour

# 1. Cargar directamente la tabla Delta de Capa Bronce
df_bronce = spark.read.table("default.bronce_energia_g5")

print(f"Total registros cargados desde Capa Bronce: {df_bronce.count()}")
display(df_bronce.limit(5))

Total registros cargados desde Capa Bronce: 63168


CodigoVariable,FechaHora,CodigoSICAgente,TipoMercado,Version,Valor,UnidadMedida,CodigoDuracion
DdaReal,2026-09-26T13:00:00.000Z,QIEC,No Regulado,TXF,2132.24,kWh,PT1H
DdaReal,2026-09-26T11:00:00.000Z,PEEC,No Regulado,TXF,324.05,kWh,PT1H
DdaReal,2026-09-26T08:00:00.000Z,DRUC,No Regulado,TXF,12179.69,kWh,PT1H
DdaReal,2026-09-26T10:00:00.000Z,RTQC,No Regulado,TXF,34587.69,kWh,PT1H
DdaReal,2026-09-26T05:00:00.000Z,TRPC,No Regulado,TXF,14.99,kWh,PT1H


In [0]:
# Función para auditar y registrar filas antes y después (requisito estricto del protocolo)
def auditar_paso(df_antes, df_despues, nombre_transformacion):
    filas_antes = df_antes.count()
    filas_despues = df_despues.count()
    diferencia = filas_antes - filas_despues
    print(f"🔹 {nombre_transformacion}")
    print(f"   Filas antes:   {filas_antes}")
    print(f"   Filas después: {filas_despues}")
    print(f"   Variación:     {diferencia} registros descartados/afectados\n")

# T1: Deduplicación exacta de filas
df_t1 = df_bronce.dropDuplicates()
auditar_paso(df_bronce, df_t1, "T1: Deduplicación de registros duplicados")

# T2: Descarte de nulos en columnas críticas (FechaHora, Valor, Agente)
df_t2 = df_t1.dropna(subset=["FechaHora", "Valor", "CodigoSICAgente"])
auditar_paso(df_t1, df_t2, "T2: Descarte de valores nulos")

# T3: Estandarización categórica (Mayúsculas y trim en TipoMercado)
df_t3 = df_t2.withColumn("TipoMercado", trim(upper(col("TipoMercado"))))
auditar_paso(df_t2, df_t3, "T3: Normalización de TipoMercado (UPPER & TRIM)")

# T4: Tipificación y separación temporal (Fecha y Bloque Horario)
df_t4 = df_t3.withColumn("Fecha_Dia", to_date(col("FechaHora"))) \
             .withColumn("Hora_Bloque", hour(col("FechaHora")))
auditar_paso(df_t3, df_t4, "T4: Extracción temporal (Fecha_Dia y Hora_Bloque)")

# T5: Filtrado de anomalías operativas (Demanda <= 0 kWh)
df_t5 = df_t4.filter(col("Valor") > 0)
auditar_paso(df_t4, df_t5, "T5: Filtrado de valores físicamente inválidos (<= 0 kWh)")

# T6: Filtrado de versiones oficiales de liquidación XM (TX1 a TXF)
df_t6 = df_t5.filter(col("Version").isin(["TX1", "TX2", "TX3", "TX4", "TXF"]))
auditar_paso(df_t5, df_t6, "T6: Filtrado de versiones oficiales de liquidación (TX)")

# Asignación final
df_plata = df_t6

🔹 T1: Deduplicación de registros duplicados
   Filas antes:   63168
   Filas después: 63168
   Variación:     0 registros descartados/afectados

🔹 T2: Descarte de valores nulos
   Filas antes:   63168
   Filas después: 63168
   Variación:     0 registros descartados/afectados

🔹 T3: Normalización de TipoMercado (UPPER & TRIM)
   Filas antes:   63168
   Filas después: 63168
   Variación:     0 registros descartados/afectados

🔹 T4: Extracción temporal (Fecha_Dia y Hora_Bloque)
   Filas antes:   63168
   Filas después: 63168
   Variación:     0 registros descartados/afectados

🔹 T5: Filtrado de valores físicamente inválidos (<= 0 kWh)
   Filas antes:   63168
   Filas después: 61648
   Variación:     1520 registros descartados/afectados

🔹 T6: Filtrado de versiones oficiales de liquidación (TX)
   Filas antes:   61648
   Filas después: 46229
   Variación:     15419 registros descartados/afectados



In [0]:
# Guardar en Delta Lake (Capa Plata)
nombre_tabla_plata = "default.plata_energia_g5"

df_plata.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(nombre_tabla_plata)

print(f"🚀 ¡Tabla Delta '{nombre_tabla_plata}' persistida con éxito en Capa Plata!")
display(spark.read.table(nombre_tabla_plata).limit(5))

🚀 ¡Tabla Delta 'default.plata_energia_g5' persistida con éxito en Capa Plata!


CodigoVariable,FechaHora,CodigoSICAgente,TipoMercado,Version,Valor,UnidadMedida,CodigoDuracion,Fecha_Dia,Hora_Bloque
DdaReal,2026-09-26T05:00:00.000Z,NEUC,NO REGULADO,TXF,48517.46,kWh,PT1H,2026-09-26,5
DdaReal,2026-09-26T06:00:00.000Z,TENC,REGULADO,TXF,7970.89,kWh,PT1H,2026-09-26,6
DdaReal,2026-09-26T14:00:00.000Z,CMMC,NO REGULADO,TXF,120299.6,kWh,PT1H,2026-09-26,14
DdaReal,2026-09-26T03:00:00.000Z,BIAC,NO REGULADO,TXF,22386.46,kWh,PT1H,2026-09-26,3
DdaReal,2026-09-26T23:00:00.000Z,EXIC,NO REGULADO,TXF,10509.13,kWh,PT1H,2026-09-26,23
